In [1]:
# Pyomo is an algebraic modeling language for Python. It lets us describe
# optimization problems (sets, parameters, variables, constraints, objective)
# in a solver-agnostic way, then hand the assembled model to any compatible
# solver (GLPK, CBC, HiGHS, Gurobi, CPLEX, ...).
import pyomo.environ as pyo

Sets

$\mathcal{F} = \{$foods$\}$ \
$\mathcal{N} = \{$nutrients$\}$

Parameters

$p_i$ price for food option $i \in \mathcal{F}$ \
$r_j$ nutrition requirement for nutrient $j \in \mathcal{N}$ \
$D_{ij}$ nutrition info for food $i \in \mathcal{F}$ and nutrient $j \in \mathcal{N}$

Variables

$x_i$ amount of food $i \in \mathcal{F}$ eaten or purchased

Objective and Constraints

\begin{gather}
 \min_x \sum_{i \in \mathcal{F}}x_i p_i \;\;\;\textrm{(cost)}\\
 \textrm{s.t.} \,\, \sum_{i \in \mathcal{F}} D_{ij} \; x_i \; \ge \; r_j \;\; \forall j \in \mathcal{N} \;\;\; \textrm{(nutrient minimums)} \\ 
 x_i \; \ge \; 0 \;\; \forall i \in \mathcal{F} \;\;\; \textrm{(lower bounds)}
\end{gather}

In [2]:
# Problem instance data. The five keys parallel the math formulation above:
#   foods      -> elements of the set F
#   nutrients  -> elements of the set N (Protein, Carbs, Fat, Vitamins)
#   needs      -> r_j, minimum required amount of each nutrient
#   price      -> pi_i, cost per unit of each food
#   content    -> D_{i,j}, amount of nutrient j in one unit of food i
data = {
    # Set F: foods we can choose from
    "foods": ["fruit", "vegetables", "meat", "bread", "pasta", "eggs"],

    # Set N: the four nutrients we track
    "nutrients": ["P", "C", "F", "V"],

    # Right-hand side of each "be healthy" constraint:
    # minimum amount of each nutrient required
    "needs": {
        "P": 20,
        "C": 40,
        "F": 15,
        "V": 20,
    },

    # Objective coefficients: cost per unit of each food
    "price": {
        "fruit": 2,
        "vegetables": 2,
        "meat": 6,
        "bread": 2,
        "pasta": 3,
        "eggs": 4,
    },

    # Constraint coefficient matrix: nutrient j supplied by one unit of food i.
    # Keyed by (food, nutrient) tuples; Pyomo Param accepts this dict shape
    # directly when the parameter is indexed by two sets.
    "content": {
        ("fruit", "P"): 1,       ("fruit", "C"): 4,       ("fruit", "F"): 0,       ("fruit", "V"): 5,
        ("vegetables", "P"): 2,  ("vegetables", "C"): 3,  ("vegetables", "F"): 0,  ("vegetables", "V"): 6,
        ("meat", "P"): 8,        ("meat", "C"): 0,        ("meat", "F"): 5,        ("meat", "V"): 1,
        ("bread", "P"): 2,       ("bread", "C"): 6,       ("bread", "F"): 1,       ("bread", "V"): 1,
        ("pasta", "P"): 3,       ("pasta", "C"): 8,       ("pasta", "F"): 1,       ("pasta", "V"): 0,
        ("eggs", "P"): 6,        ("eggs", "C"): 1,        ("eggs", "F"): 4,        ("eggs", "V"): 2,
    }
}

In [3]:
# Build the model.
# ConcreteModel binds parameter values at construction time (as opposed to
# AbstractModel, which is parameterized and only instantiated later).
m = pyo.ConcreteModel()

# Sets become indexing structures for parameters, variables, and constraints.
# `initialize` accepts any iterable of set members.
m.FOOD = pyo.Set(initialize = data["foods"])
m.NUTRIENTS = pyo.Set(initialize = data["nutrients"])

# Parameters are constant values, indexed by one or more sets. Pyomo checks
# that the initialization dict's keys match the declared set domain. The
# `content` parameter is two-dimensional: one value per (food, nutrient).
m.needs = pyo.Param(m.NUTRIENTS, initialize = data["needs"])
m.content = pyo.Param(m.FOOD, m.NUTRIENTS, initialize = data["content"])
m.price = pyo.Param(m.FOOD, initialize = data["price"])

# Decision variables: how much of each food to eat.
# domain=NonNegativeReals enforces the x_i >= 0 bound from the formulation.
# initialize=1 sets the starting value -- not required for LP solvers, but
# useful as a warm start for some solvers and for debugging.
m.eaten = pyo.Var(m.FOOD,initialize=1,domain=pyo.NonNegativeReals)

# Constraints. In Pyomo, a constraint rule is a Python function that returns
# an expression. When passed to Constraint with an indexing set, Pyomo calls
# the rule once per index value -- here, once per nutrient n -- producing one
# inequality per nutrient. This implements the "be healthy" constraints:
#     sum_i D_{i,n} * x_i  >=  r_n   for all n in N
def need_def(m,n):
    return sum(m.content[f,n] * m.eaten[f] for f in m.FOOD) >= m.needs[n]
m.need_constraint = pyo.Constraint(m.NUTRIENTS, rule = need_def)

# Objective is a single expression. sense=minimize tells the solver to push
# the expression downward. Total cost = sum over foods of (amount * price).
m.cost = pyo.Objective(expr=sum(m.eaten[f] * m.price[f] for f in m.FOOD),sense=pyo.minimize)

# pprint dumps the fully expanded model -- every set member, parameter value,
# variable, constraint, and the objective -- in a human-readable form. This
# is the easiest way to verify the model was assembled the way you intended
# before handing it to a solver.
m.pprint()

2 Set Declarations
    FOOD : Size=1, Index=None, Ordered=Insertion
        Key  : Dimen : Domain : Size : Members
        None :     1 :    Any :    6 : {'fruit', 'vegetables', 'meat', 'bread', 'pasta', 'eggs'}
    NUTRIENTS : Size=1, Index=None, Ordered=Insertion
        Key  : Dimen : Domain : Size : Members
        None :     1 :    Any :    4 : {'P', 'C', 'F', 'V'}

3 Param Declarations
    content : Size=24, Index=FOOD*NUTRIENTS, Domain=Any, Default=None, Mutable=False
        Key                 : Value
             ('bread', 'C') :     6
             ('bread', 'F') :     1
             ('bread', 'P') :     2
             ('bread', 'V') :     1
              ('eggs', 'C') :     1
              ('eggs', 'F') :     4
              ('eggs', 'P') :     6
              ('eggs', 'V') :     2
             ('fruit', 'C') :     4
             ('fruit', 'F') :     0
             ('fruit', 'P') :     1
             ('fruit', 'V') :     5
              ('meat', 'C') :     0
              ('

In [4]:
# SolverFactory looks up an installed solver by name and returns a wrapper.
# 'glpk' is the GNU Linear Programming Kit -- free, fast for moderate LPs.
# The solver binary must be on PATH
solver = pyo.SolverFactory('glpk')

# Solve the model.
# tee=True streams the solver's stdout into the notebook so you can watch its
# progress -- iteration count, objective value at each step, and termination
# status ("OPTIMAL LP SOLUTION FOUND" if all goes well).
results = solver.solve(m, tee=True)

# After solve(), Pyomo stores the solution back on the variable objects.
# pyo.value(...) extracts the numeric value out of a Var or any Pyomo
# expression. Print the optimal amount of each food, rounded to 2 decimals.
for i in data["foods"]:
    print(i , round(pyo.value(m.eaten[i])  ,2 ))

GLPSOL--GLPK LP/MIP Solver 5.0
Parameter(s) specified in the command line:
 --write C:\Users\Devin\AppData\Local\Temp\tmpq952j0vd.glpk.raw --wglp C:\Users\Devin\AppData\Local\Temp\tmp91hblbdi.glpk.glp
 --cpxlp C:\Users\Devin\AppData\Local\Temp\tmpy3d0cvp5.pyomo.lp
Reading problem data from 'C:\Users\Devin\AppData\Local\Temp\tmpy3d0cvp5.pyomo.lp'...
4 rows, 6 columns, 20 non-zeros
53 lines were read
Writing problem data to 'C:\Users\Devin\AppData\Local\Temp\tmp91hblbdi.glpk.glp'...
43 lines were written
GLPK Simplex Optimizer 5.0
4 rows, 6 columns, 20 non-zeros
Preprocessing...
4 rows, 6 columns, 20 non-zeros
Scaling...
 A: min|aij| =  1.000e+00  max|aij| =  8.000e+00  ratio =  8.000e+00
Problem data seem to be well scaled
Constructing initial basis...
Size of triangular part is 4
      0: obj =   0.000000000e+00 inf =   9.500e+01 (4)
      4: obj =   3.088235294e+01 inf =   0.000e+00 (0)
*     7: obj =   2.371212121e+01 inf =   0.000e+00 (0)
OPTIMAL LP SOLUTION FOUND
Time used:   0.0 s